In [1]:
 # Check GPU type
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# Install ultralytics
!pip -q install  ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.2/881.2 kB 8.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')


!ls /content/drive/MyDrive/Colab\ Notebooks/


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Import libraries
import pandas as pd
import os
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm
import cv2
import yaml
import matplotlib.pyplot as plt
from ultralytics import YOLO
import multiprocessing

In [ ]:
# Path to where your data is stored
DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks')

# Preview data files available
os.listdir(DATA_DIR)

In [ ]:
# Set up directoris for training a yolo model

# Images directories
DATASET_DIR = Path('datasets/dataset')
IMAGES_DIR = DATASET_DIR / 'images'
TRAIN_IMAGES_DIR = IMAGES_DIR / 'train'
VAL_IMAGES_DIR = IMAGES_DIR / 'val'
TEST_IMAGES_DIR = IMAGES_DIR / 'test'

# Labels directories
LABELS_DIR = DATASET_DIR / 'labels'
TRAIN_LABELS_DIR = LABELS_DIR / 'train'
VAL_LABELS_DIR = LABELS_DIR / 'val'
TEST_LABELS_DIR = LABELS_DIR / 'test'

In [ ]:
# Unzip images to 'images' dir
shutil.unpack_archive(DATA_DIR / 'images.zip', 'images')

In [ ]:
# Load train and test files
train = pd.read_csv(DATA_DIR / 'Train.csv')
test = pd.read_csv(DATA_DIR / 'Test.csv')
ss = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')

# Add an image_path column
train['image_path'] = [Path('images/' + x) for x in train.Image_ID]
test['image_path'] = [Path('images/' + x) for x in test.Image_ID]

# Map str classes to ints (label encoding targets)
class_mapper = {x:y for x,y in zip(sorted(train['class'].unique().tolist()), range(train['class'].nunique()))}
train['class_id'] = train['class'].map(class_mapper)

# Preview the head of the train set
train.head()

In [ ]:
test.head()

In [ ]:
ss.head()

In [ ]:
# Split data into training and validation
train_unique_imgs_df = train.drop_duplicates(subset = ['Image_ID'], ignore_index = True)
X_train, X_val = train_test_split(train_unique_imgs_df, test_size = 0.25, stratify=train_unique_imgs_df['class'], random_state=42)

X_train = train[train.Image_ID.isin(X_train.Image_ID)]
X_val = train[train.Image_ID.isin(X_val.Image_ID)]

# Check shapes of training and validation data
X_train.shape, X_val.shape

In [ ]:
# Preview target distribution, seems there a class imbalance that needs to be handled
X_train['class'].value_counts(normalize = True), X_val['class'].value_counts(normalize = True)

In [ ]:
# Check if dirs exist, if they do, remove them, otherwise create them.
# This only needs to run once
for DIR in [TRAIN_IMAGES_DIR,VAL_IMAGES_DIR, TEST_IMAGES_DIR, TRAIN_LABELS_DIR,VAL_LABELS_DIR,TEST_LABELS_DIR]:
  if DIR.exists():
    shutil.rmtree(DIR)
  DIR.mkdir(parents=True, exist_ok = True)

In [ ]:
# Copy train, val and test images to their respective dirs
for img in tqdm(X_train.image_path.unique()):
  shutil.copy(img, TRAIN_IMAGES_DIR / img.parts[-1])

for img in tqdm(X_val.image_path.unique()):
  shutil.copy(img, VAL_IMAGES_DIR / img.parts[-1])

for img in tqdm(test.image_path.unique()):
  shutil.copy(img, TEST_IMAGES_DIR / img.parts[-1])

In [ ]:
X_train.head()

In [ ]:
import multiprocessing
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm
import shutil
import pandas as pd

# Function to convert the bboxes to YOLO format
def convert_to_yolo(bbox, width, height):
    ymin, xmin, ymax, xmax = bbox['ymin'], bbox['xmin'], bbox['ymax'], bbox['xmax']
    class_id = bbox['class_id']

    # Normalize the coordinates
    x_center = (xmin + xmax) / 2 / width
    y_center = (ymin + ymax) / 2 / height
    bbox_width = (xmax - xmin) / width
    bbox_height = (ymax - ymin) / height

    return f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}"

# Top-level function to save annotations for a single image
def save_yolo_annotations_task(task):
    image_path, bboxes, output_dir = task
    try:
        img = np.array(Image.open(str(image_path)))
        height, width, _ = img.shape
    except Exception as e:
        print(f"Error opening image {image_path}: {e}")
        return

    label_file = Path(output_dir) / f"{Path(image_path).stem}.txt"
    with open(label_file, 'w') as f:
        for bbox in bboxes:
            annotation = convert_to_yolo(bbox, width, height)
            f.write(f"{annotation}\n")

# Function to clear output directory
def clear_output_dir(output_dir):
    if Path(output_dir).exists():
        shutil.rmtree(output_dir)
    Path(output_dir).mkdir(parents=True, exist_ok=True)

# Function to process the dataset and save annotations
def process_dataset(dataframe, output_dir):
    # Clear the output directory to prevent duplicate annotations
    clear_output_dir(output_dir)

    # Group the DataFrame by 'image_path'
    grouped = dataframe.groupby('image_path')
    tasks = [(image_path, group.to_dict('records'), output_dir) for image_path, group in grouped]

    # Use multiprocessing Pool to process tasks
    with multiprocessing.Pool() as pool:
        list(tqdm(pool.imap_unordered(save_yolo_annotations_task, tasks), total=len(tasks)))


# Save train and validation labels to their respective dirs
process_dataset(X_train, TRAIN_LABELS_DIR)
process_dataset(X_val, VAL_LABELS_DIR)

In [ ]:
# Train images dir
TRAIN_IMAGES_DIR

In [ ]:
import yaml

# Create a data.yaml file required by YOLO
class_names = sorted(train['class'].unique().tolist())
num_classes = len(class_names)

data_yaml = {
    # Data configuration for YOLO model
    'train': '/content/' + str(TRAIN_IMAGES_DIR),
    'val': '/content/' + str(VAL_IMAGES_DIR),
    'test': '/content/' + str(TEST_IMAGES_DIR),
    'nc': num_classes,
    'names': class_names,

    # Augmentation settings
    'augmentation': {
        'mosaic': True,            # Mosaic augmentation (combining 4 images)
        'mixup': 0.1,              # Mixup with a probability of 0.1
        'flip': True,              # Enable horizontal flip
        'hsv_h': 0.015,            # Adjust hue
        'hsv_s': 0.7,              # Adjust saturation
        'hsv_v': 0.4,              # Adjust value (brightness)
        'scale': [0.5, 1.5],       # Scale between 50% and 150%
        'translate': 0.1,          # Translate images by up to 10%
        'shear': 0.01,             # Minor shearing effect
        'perspective': 0.0         # Perspective transformation
    },

    # Optional settings for fine-tuning (uncomment if needed)
    # 'lr0': 0.001,               # Initial learning rate
    # 'lrf': 0.01                 # Final learning rate
}

# Write data to a YAML file
yaml_path = 'data.yaml'
with open(yaml_path, 'w') as file:
    yaml.dump(data_yaml, file, default_flow_style=False)

# Preview data yaml file content
data_yaml


In [ ]:
# Plot some images and their bboxes to ensure the conversion was done correctly
def load_annotations(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
    boxes = []
    for line in lines:
        class_id, x_center, y_center, width, height = map(float, line.strip().split())
        boxes.append((class_id, x_center, y_center, width, height))
    return boxes

# Function to plot an image with its bounding boxes
def plot_image_with_boxes(image_path, boxes):
    # Load the image
    image = np.array(Image.open(str(image_path)))


    # Get image dimensions
    h, w, _ = image.shape

    # Plot the image
    plt.figure(figsize=(10, 10))
    plt.imshow(image)

    # Plot each bounding box
    for box in boxes:
        class_id, x_center, y_center, width, height = box
        # Convert YOLO format to corner coordinates
        xmin = int((x_center - width / 2) * w)
        ymin = int((y_center - height / 2) * h)
        xmax = int((x_center + width / 2) * w)
        ymax = int((y_center + height / 2) * h)

        # Draw the bounding box
        plt.gca().add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                          edgecolor='red', facecolor='none', linewidth=2))
        plt.text(xmin, ymin - 10, f'Class {int(class_id)}', color='red', fontsize=8, weight='bold')

    plt.axis('off')
    plt.show()

# Directories for images and labels
IMAGE_DIR = TRAIN_IMAGES_DIR
LABEL_DIR = TRAIN_LABELS_DIR

# Plot a few images with their annotations
for image_name in os.listdir(IMAGE_DIR)[:5]:
    image_path = IMAGE_DIR / image_name
    label_path = LABEL_DIR / (image_name.replace('.jpg', '.txt').replace('.png', '.txt'))

    if label_path.exists():
        boxes = load_annotations(label_path)
        print(f"Plotting {image_name} with {len(boxes)} bounding boxes.")
        plot_image_with_boxes(image_path, boxes)
    else:
        print(f"No annotations found for {image_name}.")


In [ ]:
# Load a yolo pretrained model
model = YOLO('/content/drive/MyDrive/Colab Notebooks/yolo_model_greater_epochs_higer_image and 11s.pt')



In [ ]:
# Validate the model on the validation set
results = model.val()

In [ ]:
# Assuming 'model' is your trained YOLO model
#model.save('/content/yolo_model_more_epochs_higer_image_AND_MODRE_FINETUNING.pt')  # Save model as .pt file

#from google.colab import files
#files.download('/content/yolo_model_more_epochs_higer_image_AND_MODRE_FINETUNING.pt')


In [ ]:
from ultralytics import YOLO

# Load the saved model
model = YOLO('/content/drive/MyDrive/Colab Notebooks/yolo_model_greater_epochs_higer_image and 11s.pt')  # Specify the path where the model is saved


In [ ]:
# Load the trained YOLO model
model=model

# Load test data from test.csv
test = pd.read_csv(DATA_DIR / 'Test.csv')
test_dir_path = '/content/datasets/dataset/images/test'

# Initialize an empty list to store results
all_data = []
max_attempts = 5  # Define the maximum number of attempts per image
no_detection_count = 0  # Counter for images with no detections after all attempts

# Iterate through each image listed in test.csv
for image_id in tqdm(test.Image_ID):
    img_path = os.path.join(test_dir_path, image_id)

    # Check if the image exists before running prediction
    if os.path.exists(img_path):
        attempt = 0
        detected = False

        # Retry loop
        while attempt < max_attempts and not detected:
            attempt += 1
            results = model(img_path)

            # Check if there are any detections
            if results[0].boxes:
                boxes = results[0].boxes.xyxy.tolist()
                classes = results[0].boxes.cls.tolist()
                confidences = results[0].boxes.conf.tolist()
                names = results[0].names

                # Append actual detections to the results list
                for box, cls, conf in zip(boxes, classes, confidences):
                    x1, y1, x2, y2 = box
                    detected_class = names[int(cls)]
                    all_data.append({
                        'Image_ID': image_id,
                        'class': detected_class,
                        'confidence': conf,
                        'ymin': y1,
                        'xmin': x1,
                        'ymax': y2,
                        'xmax': x2
                    })
                detected = True  # Stop further attempts for this image once detected
            else:
                print(f"Attempt {attempt} for {image_id} yielded no detections.")

        # If still no detection after max attempts
        if not detected:
            no_detection_count += 1  # Increment the no-detection counter
            all_data.append({
                'Image_ID': image_id,
                'class': '',  # Or keep blank if preferred
                'confidence': 0.0,
                'ymin': None,
                'xmin': None,
                'ymax': None,
                'xmax': None
            })
    else:
        print(f"Warning: Image {image_id} does not exist in the directory {test_dir_path}")

# Convert to DataFrame
sub = pd.DataFrame(all_data)

# Save to submission.csv
sub.to_csv('submission11s50EPOCH1400SIZE.csv', index=False)

# Print the total number of images with no detections
print(f"Total number of images with no detections after {max_attempts} attempts: {no_detection_count}")


In [ ]:
from google.colab import files
files.download('submission11s50EPOCH1400SIZE.csv')